In [1]:
import pandas as pd
clean_data = pd.read_csv('../data/clean_data.csv')
clean_data.head()

,unit_number,time_in_cycles,op_settings_1,op_settings_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
1,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
2,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
3,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044
4,1,6,-0.0043,-0.0001,642.10,1584.47,1398.37,21.61,554.67,2388.02,9049.68,47.16,521.68,2388.03,8132.85,8.4108,391,38.98,23.3669


In [2]:
cycles_per_engine = clean_data.groupby('unit_number')['time_in_cycles'].max()
max = cycles_per_engine.reset_index()
max = max.rename(columns={'time_in_cycles': 'max_cycles'})

max.head()

,unit_number,max_cycles
0,1,192
1,2,287
2,3,179
3,4,189
4,5,269


In [3]:

labeled_data = clean_data.merge(max, on='unit_number', how='left')
labeled_data.head()

,unit_number,time_in_cycles,op_settings_1,op_settings_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,max_cycles
0,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,192
1,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,192
2,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,192
3,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,192
4,1,6,-0.0043,-0.0001,642.10,1584.47,1398.37,21.61,554.67,2388.02,9049.68,47.16,521.68,2388.03,8132.85,8.4108,391,38.98,23.3669,192


In [4]:
labeled_data["will_fail_within_30_cycles"] = (labeled_data["max_cycles"] - labeled_data["time_in_cycles"]) <= 30
labeled_data.head()

,unit_number,time_in_cycles,op_settings_1,op_settings_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,max_cycles,will_fail_within_30_cycles
0,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,...,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,192,False
1,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,...,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,192,False
2,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,...,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,192,False
3,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,...,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,192,False
4,1,6,-0.0043,-0.0001,642.10,1584.47,1398.37,21.61,554.67,2388.02,...,47.16,521.68,2388.03,8132.85,8.4108,391,38.98,23.3669,192,False


In [5]:
sensor_names =["op_settings_1", "op_settings_2"] + [f'sensor_{i}' for i in range(1, 22) if f'sensor_{i}' in labeled_data.columns]  
X = labeled_data[sensor_names]
y = labeled_data['will_fail_within_30_cycles']
groups = labeled_data["unit_number"]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
gkf = GroupKFold(n_splits=5)
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


# X = sensor features (unit_number NOT included)
# y = binary labels (1 = failing, 0 = healthy)
# groups = the unit_number column

scale = StandardScaler()
precision_total = []
recall_total = []
f1_total = []
roc_total = []



for train_idx, val_idx in gkf.split(X, y, groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_train_scaled = scale.fit_transform(X_train)   # fit + transform
    X_val_scaled   = scale.transform(X_val) 
    clf = LogisticRegression().fit(X_train_scaled, y_train)
    y_predict = clf.predict(X_val_scaled)
    y_proba = clf.predict_proba(X_val_scaled)[:, 1]
    precision = precision_score(y_val, y_predict)
    recall = recall_score(y_val, y_predict)
    f1  = f1_score(y_val, y_predict)
    roc = roc_auc_score(y_val, y_proba)
    precision_total.append(precision)
    recall_total.append(recall)
    f1_total.append(f1)
    roc_total.append(roc)


print("Precision:", sum(precision_total) / len(precision_total))
print("Recall:   ", sum(recall_total) / len(recall_total))
print("F1:       ", sum(f1_total) / len(f1_total))
print("ROC-AUC:  ", sum(roc_total) / len(roc_total))

Precision: 0.879313058367876
Recall:    0.8377419354838709
F1:        0.8570137047870772
ROC-AUC:   0.9891418235992215


In [11]:
scale = StandardScaler()
precision_total_b = []
recall_total_b = []
f1_total_b = []
roc_total_b = []



for train_idx, val_idx in gkf.split(X, y, groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_train_scaled = scale.fit_transform(X_train)   # fit + transform
    X_val_scaled   = scale.transform(X_val) 
    clf = LogisticRegression(class_weight ='balanced').fit(X_train_scaled, y_train)
    y_predict = clf.predict(X_val_scaled)
    y_proba = clf.predict_proba(X_val_scaled)[:, 1]
    precision = precision_score(y_val, y_predict)
    recall = recall_score(y_val, y_predict)
    f1  = f1_score(y_val, y_predict)
    roc = roc_auc_score(y_val, y_proba)
    precision_total_b.append(precision)
    recall_total_b.append(recall)
    f1_total_b.append(f1)
    roc_total_b.append(roc)


print("Precision:", sum(precision_total_b) / len(precision_total_b))
print("Recall:   ", sum(recall_total_b) / len(recall_total_b))
print("F1:       ", sum(f1_total_b) / len(f1_total_b))
print("ROC-AUC:  ", sum(roc_total_b) / len(roc_total_b))



Precision: 0.7286887211308303
Recall:    0.9438709677419356
F1:        0.8214359461591567
ROC-AUC:   0.9891201462370607


Baseline:

 Precision: 0.879313058367876

 Recall:    0.8377419354838709

 F1:        0.8570137047870772

 ROC-AUC:   0.9891418235992215

Balenced:

 Precision: 0.7286887211308303

 Recall:    0.9438709677419356

 F1:        0.8214359461591567
 
 ROC-AUC:   0.9891201462370607

Results: Due to the same engine appearing in the data multiple times throughout the data,
a simple train test split would result in data leakage as the same engine data could appear in both
training and test data, and this makes it too easy for the model to predict if the engine is currently 
failing or not. When setting the parameter of 'class_weight' to balanced, the model put more emphasis 
on the rarer case of failure, meaning that during training, failing to identify failure is more
punishing, so the model will flag more engines as failing, increasing recall. However this will also
decrease precision but also falsely flags more healthy engines as failing.

In [14]:
labeled_data.to_csv('../data/labeled_data.csv', index=False)
print("Saving the labeled data for usage in training and tuning the other models")

Saving the labeled data for usage in training and tuning the other models
